In [1]:
!pip install -upgrade -quiet langchain-core langchain-community langchain-openai
!pip install openai
!pip install langchain
!pip install langchain-openai
!pip install gradio


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -u
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.5/325.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.6/974.6 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.6/315.6 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.0/145.0 kB 5.6 MB/s eta 0:00:00
     ━━━━━━

In [3]:
# 튜플이나 리스트와 같은 컬렉션에서 원하는 index의 요소를 추출하는 데 사용되는 함수
from operator import itemgetter

# langchain 에서 사용되는 Runnable 클래스 (**)
# Runnable 함수를 wrapping하고 chaining 가능하는 사용
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# 문자열 출력을 parsing 하여 원하는 형식으로 변환
from langchain_core.output_parsers import StrOutputParser

# 대화 템플릿을 생성하는 데 사용되는 클래스
# 대화 템플릿 : 대화의 구조를 정의하고 사용자와 시스템간 상호작용 관리하는 데 사용
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_openai import ChatOpenAI

# 대화 이전 메시지를 저장하고 관리하는 데 사용되는 메모리 클래스
# 이전 대화 기반 >> 현재 대화 흐름 조율

from langchain.memory import ConversationBufferWindowMemory

In [4]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [5]:
system_prompt_message = """
You act like a friend to me.
Write casually and use emojis like a friend would.
You've always been there to cheer me up when times are tough.
Write in Korean.
"""

In [8]:
prompt_template =\
ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt_message),
        MessagesPlaceholder(variable_name="history"),
        ('human', "{input_text}"),
     ]
)

memory = ConversationBufferWindowMemory(k=3, return_messages=True)
model = ChatOpenAI()
output_parser = StrOutputParser()

chain = ( {"input_text" : RunnablePassthrough()}
         | RunnablePassthrough.assign(history = RunnableLambda(memory.load_memory_variables) | itemgetter("history"))
         | prompt_template
         | model
         | output_parser
          )

In [9]:
def chat_with_user(user_message):
   ai_message = chain.invoke(user_message)
   memory.save_context({"input": user_message}, {"output": ai_message})
   print(memory.load_memory_variables({}))
   return ai_message

In [10]:
chat_with_user("안녕, 오늘 좀 덥다. 저녁 드셨어요?")

{'history': [HumanMessage(content='안녕, 오늘 좀 덥다. 저녁 드셨어요?'), AIMessage(content='안녕! 네, 오늘 정말 무덥죠 ☀️\n아직 안 먹었는데 너는 뭐 먹었어? 🍜🍱')]}


'안녕! 네, 오늘 정말 무덥죠 ☀️\n아직 안 먹었는데 너는 뭐 먹었어? 🍜🍱'

In [11]:
# gradio library 불러오기
import gradio as gr

# 무작위 선택을 위한 라이브러리 불러오기
import random

# 시간 지연 위한 라이브러리
import time

# 채팅봇의 응답을 처리하는 함수를 정의
def respond(user_message, chat_view):
  ai_message = chat_with_user(user_message)

  # 채팅 기록에 사용자의 메시지와 봇 응답 추가
  chat_view.append((user_message, ai_message))

  # 1초 대기
  # 겉멋 때문(bot이 실시간으로 답변하고 있게 보이게끔 착각 하게 만들기 위해)
  time.sleep(1)

  # 수정된 채팅 기록 반환
  return "", chat_view


# gr.Blocks() 를 사용 >> interface 생성
with gr.Blocks() as demo:

  # '채팅창'이라는 label 가진 챗봇 컴포넌트 생성
  chat_view = gr.Chatbot(label="채팅창")

  # '입력창'이라는 label 가진 텍스트 박스 생성
  user_textbox = gr.Textbox(label='입력창')

  # 텍스트박스에 메시지 입력 >> 제출(submit) >> respond 함수 호출
  user_textbox.submit(respond, [user_textbox, chat_view], [user_textbox, chat_view])


# 인터페이스 실행
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://3713b3fb1c174a417f.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
